# 전체 코드

## 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install torch torchvision torchaudio transformers sentence-transformers qdrant-client langchain langchain-community langchain-openai openai scikit-learn rank_bm25 tiktoken numpy tqdm langchain-qdrant nest-asyncio

In [ ]:
# 필요한 라이브러리 임포트
import os
import asyncio
import logging
import time
import datetime
from typing import List, Dict, Any, Optional, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant as LangchainQdrant
from langchain.schema import Document
from langchain.retrievers import BM25Retriever
from openai import AsyncOpenAI
import nest_asyncio
from google.colab import userdata

## 기본 설정 및 초기화

In [ ]:
# 설정
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"
MODEL_NAME = "centwon/ko-gpt-trinity-1.2B-v0.5_v2"
MAX_GPT_USAGE = 3

# 초기화
nest_asyncio.apply()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
openai_api_key = userdata.get('FINAL_TEAM3')
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_store = LangchainQdrant(client=client, collection_name=COLLECTION_NAME, embeddings=embeddings)
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
openai_client = AsyncOpenAI(api_key=openai_api_key)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 전역 변수
gpt_usage_count = 0
last_reset_date = datetime.date.today()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10

## QdrantRetriever

In [ ]:
class QdrantRetriever:
    def __init__(self, client: QdrantClient, collection_name: str):
        self.client = client
        self.collection_name = collection_name

    def retrieve(self, query_vector: List[float], top_k: int = 20) -> List[Any]:
        return self.client.search(collection_name=self.collection_name, query_vector=query_vector, limit=top_k)


## Qdrant 검색 결과 ->  Langchain의 Document 객체 리스트로 변환

In [ ]:
def create_documents_from_qdrant_results(results) -> List[Document]:
    return [Document(
        page_content=result.payload.get('답변', 'N/A'),
        metadata={
            'id': result.id,
            '질병 카테고리': result.payload.get('질병_카테고리', 'N/A'),
            '질병': result.payload.get('질병', 'N/A'),
            '부서': result.payload.get('부서', 'N/A'),
            '의도': result.payload.get('의도', 'N/A'),
            'score': result.score
        }
    ) for result in results]

## ensemble model

In [ ]:
async def ensemble_search(query: str, top_k: int = 5) -> List[Document]:
    encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")
    qdrant_retriever = QdrantRetriever(client=client, collection_name="son99_d")

    query_vector = encoder.encode(query).tolist()
    qdrant_results = qdrant_retriever.retrieve(query_vector, top_k=20)

    if not qdrant_results:
        logging.warning("Qdrant에서 검색 결과가 없습니다.")
        return []

    documents = create_documents_from_qdrant_results(qdrant_results)
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_results = bm25_retriever.get_relevant_documents(query)

    if not bm25_results:
        logging.warning("BM25에서 유사한 문서가 없습니다.")
        return documents[:top_k]

    combined_results = []
    for bm25_result in bm25_results:
        matching_qdrant_result = next((doc for doc in documents if doc.page_content == bm25_result.page_content), None)
        if matching_qdrant_result:
            combined_score = 0.5 * matching_qdrant_result.metadata.get('score', 0) + 0.5 * bm25_result.metadata.get('score', 1)
            matching_qdrant_result.metadata['combined_score'] = combined_score
            combined_results.append(matching_qdrant_result)

    if not combined_results:
        logging.warning("결합된 결과가 없습니다.")
        return documents[:top_k]

    combined_results.sort(key=lambda x: x.metadata['combined_score'], reverse=True)
    return [result for result in combined_results[:top_k] if result.page_content and result.page_content.strip()]


## GPT 모델 사용량 체크

In [ ]:
def check_and_reset_gpt_usage():
    global gpt_usage_count, last_reset_date
    today = datetime.date.today()
    if today > last_reset_date:
        gpt_usage_count = 0
        last_reset_date = today

## GPT4o-turbo model

In [ ]:
async def generate_gpt4_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        system_message = (
            "You are an expert medical assistant designed to support seniors during medical emergencies "
            "while traveling. Your primary role is to provide accurate and clear medical advice based on the user's "
            "symptoms or questions about diseases. When the user asks about a disease or symptom, "
            "show the related metadata, such as the disease and intent, and then provide the best advice based on "
            "the retrieved information. Please respond in Korean and provide a complete response."
            "Generating an answer, please make it within max_tokens."
        )

        user_message = f"User Query: {query}\n\nContext: {context}\n\nMetadata: {metadata}"

        response = await openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=1000,
            temperature=0.3,
            n=1,
            stop=None,
        )

        generated_response = response.choices[0].message.content.strip()
        logging.info(f"GPT-4 응답 생성 완료: 길이={len(generated_response)}")
        return generated_response

    except Exception as e:
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다."


## ko-gpt-trinity-1.2B-v0.5_v2
 - 우리가 파인 튜닝한 모델 v2

In [ ]:
async def generate_custom_model_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to('cuda' if torch.cuda.is_available() else 'cpu')
        prompt = f"""질문: {query}\n\n컨텍스트: {context or ''}\n\n메타데이터: {metadata or {}}\n\n주어진 질문, 컨텍스트, 메타데이터를 바탕으로 답변을 생성해주세요. 답변은 구체적이고 정확해야 하며, 질문과 직접적으로 관련이 있어야 합니다.\n\n답변:"""

        inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=300,
                num_return_sequences=1,
                no_repeat_ngram_size=3,
                temperature=0.7,
                do_sample=True,
                top_k=50,
                top_p=0.95,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("답변:")[-1].strip()

        sentences = response.split('.')
        filtered_sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        response = '. '.join(filtered_sentences[:5])

        return response

    except Exception as e:
        logging.error(f"커스텀 모델 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다."

    finally:
        if 'model' in locals():
            del model
            torch.cuda.empty_cache()

## model 응답 생성 관리

In [ ]:
def generate_rule_based_response(query: str, context: str, metadata: Dict[str, Any]) -> str:
    disease = metadata.get('질병', '')
    intent = metadata.get('의도', '')
    category = metadata.get('질병 카테고리', '')

    response = f"{disease}에 대한 {intent}와 관련된 정보입니다.\n\n"

    if intent == '정의':
        response += f"{disease}은(는) {category}에 속하는 질병으로, {context}"
    elif intent == '증상':
        response += f"{disease}의 주요 증상은 다음과 같습니다: {context}"
    elif intent == '치료':
        response += f"{disease}의 치료 방법은 다음과 같습니다: {context}"
    elif intent == '예방':
        response += f"{disease}의 예방법은 다음과 같습니다: {context}"
    else:
        response += context

    response += f"\n\n이 정보는 {category} 분야의 전문가의 조언을 바탕으로 합니다. 더 자세한 정보나 개인적인 의료 상담이 필요하다면 전문의와 상담하시기 바랍니다."

    return response

In [ ]:
async def generate_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> Tuple[str, str]:
    global gpt_usage_count

    try:
        if gpt_usage_count < MAX_GPT_USAGE:
            gpt_usage_count += 1
            response = await generate_gpt4_response(query, context, metadata)
            return response, "GPT-4"
        else:
            response = generate_rule_based_response(query, context, metadata)
            return response, "Custom Model"
    except Exception as e:
        logging.error(f"응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다.", "Error"


## 사용자 쿼리 처리

In [ ]:
async def process_query_async(query: str) -> Tuple[str, str, float]:
    start_time = time.time()
    logging.info(f"처리 중인 쿼리: {query}")

    try:
        ensemble_results = await ensemble_search(query, 5)

        if not ensemble_results:
            logging.warning("앙상블 검색 결과가 없습니다.")
            response, model = await generate_response(query)
            return response, model, time.time() - start_time

        best_match = ensemble_results[0]
        context = best_match.page_content
        metadata = best_match.metadata

        print(f"\nDB에서 검색된 정보:\n내용: {context}\n메타데이터: {metadata}\n")

        if metadata.get('combined_score', 0) < 0.7:
            logging.warning("질문과 관련된 정보를 찾기 어렵습니다. 다른 방식으로 질문해 주시겠어요?")
            return "질문과 관련된 정보를 찾기 어렵습니다. 다른 방식으로 질문해 주시겠어요?", "System Message", time.time() - start_time

        response, model = await generate_response(query, context, metadata)

        processing_time = time.time() - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return response, model, processing_time

    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
        response, model = await generate_response(query)
        return response, model, time.time() - start_time


In [ ]:
async def run_system():
    print("의료 정보 시스템을 초기화하는 중...")

    try:
        client.get_collections()
        print("Qdrant 서버에 성공적으로 연결되었습니다.")

        global st_model
        st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
        print("임베딩 모델이 로드되었습니다.")

        if not openai_api_key:
            raise ValueError("OpenAI API 키가 설정되지 않았습니다.")
        print("OpenAI API 키가 확인되었습니다.")

        global tokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        print("커스텀 모델 토크나이저가 초기화되었습니다.")

        print("\n의료 정보 시스템이 성공적으로 초기화되었습니다.")
        print("시스템이 준비되었습니다. 질문을 입력해주세요.")

    except Exception as e:
        print(f"시스템 초기화 중 오류가 발생했습니다: {str(e)}")
        print("시스템을 종료합니다.")
        sys.exit(1)

## 성능 테스트

In [ ]:
async def evaluate_system():
    """
    시스템의 성능을 평가하는 함수입니다.
    여러 테스트 쿼리를 처리하고 결과를 출력합니다.
    """
    test_queries = [
        "순환기 질환인 고혈압 증상이 궁금해요",
        "고혈압 치료방법이 궁금해요",
        "고혈압에 대해 알고 싶어요",
        "머리가 어지럽고 가슴통증이 있고 시야에 문제가 있고 코피가나",
        "아 어지러운데 이게뭐지"
    ]

    total_time = 0
    total_queries = len(test_queries)

    for query in test_queries:
        try:
            response, model, processing_time = await process_query_async(query)
            total_time += processing_time

            print(f"질문: {query}")
            print(f"답변 ({model}): {response}")
            print(f"처리 시간: {processing_time:.2f}초")
            print("-" * 50)

        except Exception as e:
            logging.error(f"쿼리 처리 중 오류 발생: {str(e)}")

    avg_time = total_time / total_queries if total_queries > 0 else 0
    print(f"평균 처리 시간: {avg_time:.2f}초")


In [ ]:
async def main():
    await run_system()

    while True:
        user_query = input("질문을 입력하세요 (종료하려면 'quit' 입력): ")
        if user_query.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        try:
            response, model, processing_time = await process_query_async(user_query)
            print(f"\n답변 ({model}):\n{response}")
            print(f"처리 시간: {processing_time:.2f}초\n")
        except Exception as e:
            print(f"오류 발생: {str(e)}")
            print("다시 질문해 주세요.")

if __name__ == "__main__":
    asyncio.run(main())

의료 정보 시스템을 초기화하는 중...
Qdrant 서버에 성공적으로 연결되었습니다.
임베딩 모델이 로드되었습니다.
OpenAI API 키가 확인되었습니다.
커스텀 모델 토크나이저가 초기화되었습니다.

의료 정보 시스템이 성공적으로 초기화되었습니다.
시스템이 준비되었습니다. 질문을 입력해주세요.
질문을 입력하세요 (종료하려면 'quit' 입력): 고혈압 증상에 대해서 알려줘.


/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(



DB에서 검색된 정보:
내용: 고혈압을 진단하기 위해서는 증상을 관찰하고, 전문의의 조언을 따라야 합니다. 만약 고혈압이 의심된다면, 혈압을 측정하기 위해 병원을 방문하거나 혈압계를 사용해야 합니다. 고혈압 환자는 정기적으로 혈압을 평가하고 변화를 주시해야 합니다.
메타데이터: {'id': 1814, '질병 카테고리': '순환기질환', '질병': '고혈압', '부서': '내과', '의도': '정의', 'score': 0.7576843, 'combined_score': 0.7576843}


답변 (GPT-4):
고혈압은 혈압이 지속적으로 높은 상태를 의미합니다. 혈압이란 심장이 혈액을 동맥으로 밀어내는 힘을 말하며, 이 힘이 지나치게 강하면 동맥벽에 손상을 줄 수 있습니다. 고혈압은 "조용한 살인자"로 불리기도 하는데, 초기에는 특별한 증상이 없어 자신이 고혈압인지 모르는 경우가 많습니다. 그러나 고혈압이 장기간 지속되면 심장, 뇌, 신장 및 다른 장기에 심각한 손상을 줄 수 있습니다.

고혈압의 몇 가지 일반적인 증상은 다음과 같습니다:
- 두통
- 어지러움
- 가슴 통증
- 호흡곤란
- 시야 흐림
- 코피
- 심한 경우, 심장마비나 뇌졸중으로 이어질 수 있습니다.

이러한 증상 중 하나라도 경험한다면, 즉시 의사의 진료를 받아야 합니다. 고혈압은 혈압계를 통해 측정할 수 있으며, 정상 혈압 범위를 초과하는 경우 고혈압으로 진단될 수 있습니다. 고혈압의 관리는 생활 습관의 변화, 약물 치료, 정기적인 혈압 모니터링을 포함할 수 있습니다. 고혈압이 의심되거나 이미 진단받았다면, 정기적으로 혈압을 측정하고 의사의 지시에 따라 관리하는 것이 중요합니다.
처리 시간: 22.43초

질문을 입력하세요 (종료하려면 'quit' 입력): 식중독 증상에 대해서 알려줘.

DB에서 검색된 정보:
내용: 식중독을 진단하기 위해서는 식중독의 증상과 검사가 필요합니다. 증상을 평가하기 위해 환자의 증상에 대한 문진 및 내시경 검사가 수행됩니다. 또한, 환자의 증상에 대한 조사 


DB에서 검색된 정보:
내용: 두통은 많은 사람들이 경험하는 흔한 증상이지만, 실제로는 심각한 경우가 많습니다.
메타데이터: {'id': 8483, '질병 카테고리': '응급질환', '질병': '두통', '부서': '신경과', '의도': '원인', 'score': 0.5942968, 'combined_score': 0.5942968}


답변 (System Message):
질문과 관련된 정보를 찾기 어렵습니다. 다른 방식으로 질문해 주시겠어요?
처리 시간: 3.42초

질문을 입력하세요 (종료하려면 'quit' 입력): 두통 치료법에는 뭐가 있어?

DB에서 검색된 정보:
내용: 약물 치료는 두통 완화를 위해 일반적으로 사용되는 치료 방법 중 하나입니다. 그러나 증상이 심각한 경우에는 전문적인 도움이 필요할 수도 있습니다. 일반적인 두통을 완화시키기 위해 약물 치료뿐만 아니라 필요한 경우 수술적 치료도 고려될 수 있습니다.
메타데이터: {'id': 8611, '질병 카테고리': '응급질환', '질병': '두통', '부서': '신경과', '의도': '원인', 'score': 0.7318822, 'combined_score': 0.7318822}


답변 (GPT-4):
두통은 다양한 원인에 의해 발생할 수 있으며, 치료 방법도 그 원인에 따라 달라질 수 있습니다. 일반적으로 두통 치료를 위해 사용되는 방법은 다음과 같습니다:

1. **약물 치료**: 가벼운 두통의 경우, 일반적으로 사용되는 진통제(아세트아미노펜, 이부프로펜 등)가 효과적일 수 있습니다. 만성적이거나 더 심각한 두통의 경우, 의사는 특정한 처방약을 권할 수 있습니다.

2. **수술적 치료**: 두통의 원인이 특정한 물리적 문제(예: 뇌내 혈관 이상, 종양 등)에 기인하는 경우, 수술적 치료가 필요할 수 있습니다. 이는 매우 드문 경우이며, 전문의와의 상담을 통해 결정됩니다.

3. **생활 습관 변화**: 스트레스 관리, 충분한 수면, 규칙적인 운동, 건강한 식습관 등은 두통을 예방하고 관

결과 해석 :
 - GPT 답변이 끊기는 문제와 퀄리티 해결
 - custom 모델이 DB 내용을 그대로 가져오는 문제 발생.

문제 해결 방안 :
 - DB 미참고 답변 생성 기능 추가 (08번 코드 참조)
 - 21번 코드 생성 후 적용

거의 다 되었으니 좀만 힘내자!

실행 결과:

의료 정보 시스템을 초기화하는 중...
Qdrant 서버에 성공적으로 연결되었습니다.
임베딩 모델이 로드되었습니다.
OpenAI API 키가 확인되었습니다.
커스텀 모델 토크나이저가 초기화되었습니다.

의료 정보 시스템이 성공적으로 초기화되었습니다.
시스템이 준비되었습니다. 질문을 입력해주세요.
질문을 입력하세요 (종료하려면 'quit' 입력): 고혈압 증상에 대해서 알려줘.
/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(

DB에서 검색된 정보:
내용: 고혈압을 진단하기 위해서는 증상을 관찰하고, 전문의의 조언을 따라야 합니다. 만약 고혈압이 의심된다면, 혈압을 측정하기 위해 병원을 방문하거나 혈압계를 사용해야 합니다. 고혈압 환자는 정기적으로 혈압을 평가하고 변화를 주시해야 합니다.
메타데이터: {'id': 1814, '질병 카테고리': '순환기질환', '질병': '고혈압', '부서': '내과', '의도': '정의', 'score': 0.7576843, 'combined_score': 0.7576843}


답변 (GPT-4):
고혈압은 혈압이 지속적으로 높은 상태를 의미합니다. 혈압이란 심장이 혈액을 동맥으로 밀어내는 힘을 말하며, 이 힘이 지나치게 강하면 동맥벽에 손상을 줄 수 있습니다. 고혈압은 "조용한 살인자"로 불리기도 하는데, 초기에는 특별한 증상이 없어 자신이 고혈압인지 모르는 경우가 많습니다. 그러나 고혈압이 장기간 지속되면 심장, 뇌, 신장 및 다른 장기에 심각한 손상을 줄 수 있습니다.

고혈압의 몇 가지 일반적인 증상은 다음과 같습니다:
- 두통
- 어지러움
- 가슴 통증
- 호흡곤란
- 시야 흐림
- 코피
- 심한 경우, 심장마비나 뇌졸중으로 이어질 수 있습니다.

이러한 증상 중 하나라도 경험한다면, 즉시 의사의 진료를 받아야 합니다. 고혈압은 혈압계를 통해 측정할 수 있으며, 정상 혈압 범위를 초과하는 경우 고혈압으로 진단될 수 있습니다. 고혈압의 관리는 생활 습관의 변화, 약물 치료, 정기적인 혈압 모니터링을 포함할 수 있습니다. 고혈압이 의심되거나 이미 진단받았다면, 정기적으로 혈압을 측정하고 의사의 지시에 따라 관리하는 것이 중요합니다.
처리 시간: 22.43초

질문을 입력하세요 (종료하려면 'quit' 입력): 식중독 증상에 대해서 알려줘.

DB에서 검색된 정보:
내용: 식중독을 진단하기 위해서는 식중독의 증상과 검사가 필요합니다. 증상을 평가하기 위해 환자의 증상에 대한 문진 및 내시경 검사가 수행됩니다. 또한, 환자의 증상에 대한 조사 결과를 통해 특정한 유형의 식중독을 확인할 수 있습니다. 이러한 검사를 통해 식중독의 원인을 파악하고 적절한 치료를 위해 적절한 절차와 약물을 투여할 수 있습니다.
메타데이터: {'id': 12826, '질병 카테고리': '응급질환', '질병': '식중독', '부서': '가정의학과', '의도': '진단', 'score': 0.7246659, 'combined_score': 0.7246659}


답변 (GPT-4):
식중독은 섭취한 음식이나 물에 의해 발생하는 질병으로, 다양한 증상을 유발할 수 있습니다. 식중독의 주요 증상으로는 구토, 설사, 복통, 발열 등이 있으며, 심한 경우 탈수나 전해질 불균형을 초래할 수 있습니다. 식중독을 진단하기 위해서는 의사가 환자의 증상에 대한 문진을 실시하고 필요한 경우 내시경 검사를 포함한 여러 검사를 진행할 수 있습니다. 환자의 증상과 검사 결과를 통해 특정한 유형의 식중독을 확인하고, 그에 따른 적절한 치료를 진행합니다.

식중독에 걸렸을 때는 충분한 수분 섭취를 유지하고, 가벼운 식사를 하면서 몸을 회복시키는 것이 중요합니다. 심한 경우에는 병원을 방문하여 전문적인 치료를 받아야 합니다. 식중독 예방을 위해서는 음식을 섭취하기 전에 손을 깨끗이 씻고, 음식을 충분히 가열하여 섭취하는 것이 좋습니다. 또한, 유통기한이 지난 음식이나 의심스러운 음식은 섭취하지 않는 것이 바람직합니다.
처리 시간: 16.76초

질문을 입력하세요 (종료하려면 'quit' 입력): 아 어지러운데 이게뭐지
WARNING:root:질문과 관련된 정보를 찾기 어렵습니다. 다른 방식으로 질문해 주시겠어요?

DB에서 검색된 정보:
내용: 두통은 많은 사람들이 경험하는 흔한 증상이지만, 실제로는 심각한 경우가 많습니다.
메타데이터: {'id': 8483, '질병 카테고리': '응급질환', '질병': '두통', '부서': '신경과', '의도': '원인', 'score': 0.5942968, 'combined_score': 0.5942968}


답변 (System Message):
질문과 관련된 정보를 찾기 어렵습니다. 다른 방식으로 질문해 주시겠어요?
처리 시간: 3.42초

질문을 입력하세요 (종료하려면 'quit' 입력): 두통 치료법에는 뭐가 있어?

DB에서 검색된 정보:
내용: 약물 치료는 두통 완화를 위해 일반적으로 사용되는 치료 방법 중 하나입니다. 그러나 증상이 심각한 경우에는 전문적인 도움이 필요할 수도 있습니다. 일반적인 두통을 완화시키기 위해 약물 치료뿐만 아니라 필요한 경우 수술적 치료도 고려될 수 있습니다.
메타데이터: {'id': 8611, '질병 카테고리': '응급질환', '질병': '두통', '부서': '신경과', '의도': '원인', 'score': 0.7318822, 'combined_score': 0.7318822}


답변 (GPT-4):
두통은 다양한 원인에 의해 발생할 수 있으며, 치료 방법도 그 원인에 따라 달라질 수 있습니다. 일반적으로 두통 치료를 위해 사용되는 방법은 다음과 같습니다:

1. **약물 치료**: 가벼운 두통의 경우, 일반적으로 사용되는 진통제(아세트아미노펜, 이부프로펜 등)가 효과적일 수 있습니다. 만성적이거나 더 심각한 두통의 경우, 의사는 특정한 처방약을 권할 수 있습니다.

2. **수술적 치료**: 두통의 원인이 특정한 물리적 문제(예: 뇌내 혈관 이상, 종양 등)에 기인하는 경우, 수술적 치료가 필요할 수 있습니다. 이는 매우 드문 경우이며, 전문의와의 상담을 통해 결정됩니다.

3. **생활 습관 변화**: 스트레스 관리, 충분한 수면, 규칙적인 운동, 건강한 식습관 등은 두통을 예방하고 관리하는 데 도움이 될 수 있습니다.

4. **대체 요법**: 마사지, 침술, 요가, 명상 등과 같은 대체 요법이 일부 사람들에게는 두통 완화에 도움이 될 수 있습니다.

5. **물리치료**: 특히 긴장성 두통이나 경추성 두통의 경우, 물리치료가 도움이 될 수 있습니다.

두통의 원인과 유형에 따라 적절한 치료 방법이 달라질 수 있으므로, 지속적이거나 심각한 두통이 있는 경우에는 전문의와 상담하는 것이 중요합니다. 전문의는 증상을 평가하고, 필요한 검사를 진행한 후 가장 적합한 치료 방법을 제안할 것입니다.
처리 시간: 21.94초

질문을 입력하세요 (종료하려면 'quit' 입력): 고혈압 증상에 대해서 알려줘.

DB에서 검색된 정보:
내용: 고혈압을 진단하기 위해서는 증상을 관찰하고, 전문의의 조언을 따라야 합니다. 만약 고혈압이 의심된다면, 혈압을 측정하기 위해 병원을 방문하거나 혈압계를 사용해야 합니다. 고혈압 환자는 정기적으로 혈압을 평가하고 변화를 주시해야 합니다.
메타데이터: {'id': 1814, '질병 카테고리': '순환기질환', '질병': '고혈압', '부서': '내과', '의도': '정의', 'score': 0.7576843, 'combined_score': 0.7576843}


답변 (Custom Model):
고혈압에 대한 정의와 관련된 정보입니다.

고혈압은(는) 순환기질환에 속하는 질병으로, 고혈압을 진단하기 위해서는 증상을 관찰하고, 전문의의 조언을 따라야 합니다. 만약 고혈압이 의심된다면, 혈압을 측정하기 위해 병원을 방문하거나 혈압계를 사용해야 합니다. 고혈압 환자는 정기적으로 혈압을 평가하고 변화를 주시해야 합니다.

이 정보는 순환기질환 분야의 전문가의 조언을 바탕으로 합니다. 더 자세한 정보나 개인적인 의료 상담이 필요하다면 전문의와 상담하시기 바랍니다.
처리 시간: 3.46초

질문을 입력하세요 (종료하려면 'quit' 입력): 식중독 증상에 대해서 알려줘.

DB에서 검색된 정보:
내용: 식중독을 진단하기 위해서는 식중독의 증상과 검사가 필요합니다. 증상을 평가하기 위해 환자의 증상에 대한 문진 및 내시경 검사가 수행됩니다. 또한, 환자의 증상에 대한 조사 결과를 통해 특정한 유형의 식중독을 확인할 수 있습니다. 이러한 검사를 통해 식중독의 원인을 파악하고 적절한 치료를 위해 적절한 절차와 약물을 투여할 수 있습니다.
메타데이터: {'id': 12826, '질병 카테고리': '응급질환', '질병': '식중독', '부서': '가정의학과', '의도': '진단', 'score': 0.7246659, 'combined_score': 0.7246659}


답변 (Custom Model):
식중독에 대한 진단와 관련된 정보입니다.

식중독을 진단하기 위해서는 식중독의 증상과 검사가 필요합니다. 증상을 평가하기 위해 환자의 증상에 대한 문진 및 내시경 검사가 수행됩니다. 또한, 환자의 증상에 대한 조사 결과를 통해 특정한 유형의 식중독을 확인할 수 있습니다. 이러한 검사를 통해 식중독의 원인을 파악하고 적절한 치료를 위해 적절한 절차와 약물을 투여할 수 있습니다.

이 정보는 응급질환 분야의 전문가의 조언을 바탕으로 합니다. 더 자세한 정보나 개인적인 의료 상담이 필요하다면 전문의와 상담하시기 바랍니다.
처리 시간: 3.63초

질문을 입력하세요 (종료하려면 'quit' 입력): 두통 치료법에는 뭐가 있어?

DB에서 검색된 정보:
내용: 약물 치료는 두통 완화를 위해 일반적으로 사용되는 치료 방법 중 하나입니다. 그러나 증상이 심각한 경우에는 전문적인 도움이 필요할 수도 있습니다. 일반적인 두통을 완화시키기 위해 약물 치료뿐만 아니라 필요한 경우 수술적 치료도 고려될 수 있습니다.
메타데이터: {'id': 8611, '질병 카테고리': '응급질환', '질병': '두통', '부서': '신경과', '의도': '원인', 'score': 0.7318822, 'combined_score': 0.7318822}


답변 (Custom Model):
두통에 대한 원인와 관련된 정보입니다.

약물 치료는 두통 완화를 위해 일반적으로 사용되는 치료 방법 중 하나입니다. 그러나 증상이 심각한 경우에는 전문적인 도움이 필요할 수도 있습니다. 일반적인 두통을 완화시키기 위해 약물 치료뿐만 아니라 필요한 경우 수술적 치료도 고려될 수 있습니다.

이 정보는 응급질환 분야의 전문가의 조언을 바탕으로 합니다. 더 자세한 정보나 개인적인 의료 상담이 필요하다면 전문의와 상담하시기 바랍니다.
처리 시간: 3.38초

질문을 입력하세요 (종료하려면 'quit' 입력): quit
프로그램을 종료합니다.